In [ ]:
model.dense_transition_matrix()

In [ ]:
import numpy as np
from pomegranate.io import DataGenerator, SequenceGenerator, BatchedDataGenerator

In [ ]:
TARGET_LENGTHS = [8]
allele_name = ALLELES[0]
list_of_arrays = [per_allele_per_kfold_per_length_binders_train[allele_name][0][target_length] for target_length in TARGET_LENGTHS]
binders_array = np.concatenate(list_of_arrays)
sample_X = np.array([[char for char in item] for item in binders_array], dtype=object)
sequence_generator = DataGenerator(sample_X,batches_per_epoch=None, batch_size=None)

In [ ]:
# make a model
from pomegranate.hmm import check_random_state
from pomegranate import DiscreteDistribution, HiddenMarkovModel

In [ ]:
initialization_batch_size = None
if initialization_batch_size is None:
    initialization_batch_size = 1

In [ ]:
X_, labels_ = [], []
data = sequence_generator.batches()
for i in range(initialization_batch_size):
    batch = next(data)
    X_.extend(batch[0])
X_concat = np.concatenate(X_)

In [ ]:
random_state =   check_random_state(None)
n_components = 12
keys = np.unique(X_concat)
distributions = []
for i in range(n_components):
    weights = random_state.uniform(0, 1, size=len(keys))
    weights /= weights.sum()

    if X_concat.ndim == 1:
        distribution = DiscreteDistribution({key: weight for key, weight in zip(keys, weights)})
    else:
        distribution = IndependentComponentsDistribution.from_samples(X_concat,
                                                                      distributions=DiscreteDistribution)

    distributions.append(distribution)
k = n_components
transition_matrix = np.ones((k, k)) / k
start_probabilities = np.ones(k) / k

end_probabilities = None
end_probabilities = np.ones(k) / k
state_names = [f"s{i}" for i  in range(n_components)]
name = "viterbi_model"
multiple_check_input = True
model = HiddenMarkovModel.from_matrix(transition_matrix, distributions,
                        start_probabilities, state_names=state_names, name=name,
                        ends=end_probabilities)

In [ ]:
data_generator = sequence_generator
n = data_generator.shape[0]
n

In [ ]:
inertia = 0
lr_decay = 0
pseudocount = 0
transition_pseudocount = 0
emission_pseudocount = 0
use_pseudocount = False
edge_inertia = 0
distribution_inertia = 0
improvement = np.inf
total_improvement = 0
algorithm = "viterbi"

In [ ]:
model.dense_transition_matrix()

In [ ]:
model.states[0].distribution

In [ ]:
with open("state_graph.png", 'w+') as f:
    model.plot(file=f, crop_zero=True)

In [ ]:
def viterbi_step(model, iteration, last_log_probability_sum):
    step_size = None if inertia is None else 1 - ((1 - inertia) * (2 + iteration) ** -lr_decay)
    # make model from previous summaries
    model.from_summaries(step_size, pseudocount, transition_pseudocount, emission_pseudocount, use_pseudocount,edge_inertia, distribution_inertia)
    # collect new summaries
    log_probability_sum = [model.summarize(*batch, algorithm=algorithm,
                                           check_input=multiple_check_input) for batch in data_generator.batches()]
    log_probability_sum = sum([r for r in log_probability_sum if not np.isinf(r)])
    print(f"Log_prob sum was {log_probability_sum}")
    improvement = np.inf
    if iteration != 0:
        improvement = log_probability_sum - last_log_probability_sum
    iteration += 1
    print(f"improvement {improvement}")
    return log_probability_sum, improvement


In [ ]:
log_probability_sum, improvement = viterbi_step(model, 0, 0)

In [ ]:
model.dense_transition_matrix()

In [ ]:
model.states[0].distribution

In [ ]:
log_probability_sum, improvement = viterbi_step(model, 1, log_probability_sum)

In [ ]:
model.dense_transition_matrix()

In [ ]:
model.states[0].distribution

In [ ]:
log_probability_sum, improvement = viterbi_step(model, 2, log_probability_sum)

In [ ]:
model.dense_transition_matrix()

In [ ]:
model.states[0].distribution

In [ ]:
log_probability_sum, improvement = viterbi_step(model, 3, log_probability_sum)

In [ ]:
log_probability_sum, improvement = viterbi_step(model, 4, log_probability_sum)

In [ ]:
with open("state_graph.png", 'w+') as f:
    model.plot(file=f, crop_zero=True)

In [ ]:
model.dense_transition_matrix()

In [ ]:
model.states[0].distribution

In [ ]:
model.states[1].distribution

In [ ]:
model.states[2].distribution

In [ ]:
log_probability_sum = [model.summarize(*batch, algorithm=algorithm,
                                       check_input=multiple_check_input) for batch in data_generator.batches()]
log_probability_sum = sum([r for r in log_probability_sum if not np.isinf(r)])
log_probability_sum

In [ ]:
if iteration == 0:
    initial_log_probability_sum = log_probability_sum
else:
    improvement = log_probability_sum - last_log_probability_sum


iteration += 1
last_log_probability_sum = log_probability_sum

In [ ]:
improvement

In [ ]:
# iteration 3
model.from_summaries(step_size, pseudocount, transition_pseudocount, emission_pseudocount, use_pseudocount,edge_inertia, distribution_inertia)

In [ ]:
model.dense_transition_matrix()

In [ ]:
model.states[0].distribution

In [ ]:
log_probability_sum = [model.summarize(*batch, algorithm=algorithm,
                                       check_input=multiple_check_input) for batch in data_generator.batches()]
log_probability_sum = sum([r for r in log_probability_sum if not np.isinf(r)])
log_probability_sum

In [ ]:
if iteration == 0:
    initial_log_probability_sum = log_probability_sum
else:
    improvement = log_probability_sum - last_log_probability_sum


iteration += 1
last_log_probability_sum = log_probability_sum

In [ ]:
improvement

In [ ]:
model.dense_transition_matrix()

In [ ]:
model.states[0].distribution

In [ ]:
log_probability_sum = [model.summarize(*batch, algorithm=algorithm,
                                       check_input=multiple_check_input) for batch in data_generator.batches()]
log_probability_sum = sum([r for r in log_probability_sum if not np.isinf(r)])
log_probability_sum

In [ ]:
if iteration == 0:
    initial_log_probability_sum = log_probability_sum
else:
    improvement = log_probability_sum - last_log_probability_sum


iteration += 1
last_log_probability_sum = log_probability_sum

In [ ]:
improvement

In [ ]:
iteration = 0
improvement = np.inf
total_improvement = 0
min_iterations = 1
while improvement > stop_threshold or iteration < min_iterations + 1:
    step_size = None if inertia is None else 1 - ((1 - inertia) * (2 + iteration) ** -lr_decay)
    model.from_summaries(step_size, pseudocount, transition_pseudocount,
                         emission_pseudocount, use_pseudocount,
                         edge_inertia, distribution_inertia)
    log_probability_sum = sum(model.summarize(*batch, algorithm=algorithm,
                                              check_input=True) for batch in data_generator.batches())

    if iteration == 0:
        initial_log_probability_sum = log_probability_sum
    else:
        improvement = log_probability_sum - last_log_probability_sum
        total_improvement += improvement
    iteration += 1
    last_log_probability_sum = log_probability_sum
    print(f"Improvement {improvement} Total improvement {total_improvement}")

model.clear_summaries()
for k in range(len(model.states)):
    for l in range(model.out_edge_count[k], model.out_edge_count[k+1]):
        li = model.out_transitions[l]
        prob = model.out_transition_log_probabilities[l]
        model.graph[model.states[k]][model.states[li]]['probability'] = prob

In [ ]:
from hmm_logic_methods import load_model

In [ ]:
model = load_model(TARGET_PATH_TO_RESULTS, "0_run-8-mer-12-st-viterbi-0.0-lr-None-inertia-10-batches_model-HLA_A_02_01-0")

In [ ]:
len(sequence_generator), sequence_generator.batch_size, sequence_generator.batches_per_epoch

In [ ]:
batches_counter = 0
total_batches_counter = 0
epoch_counter = 0
entries_counter = 0

for batch in sequence_generator.batches():
    batch = batch
    entries_counter += len(batch[0])
    batches_counter+=1
    total_batches_counter += 1


In [ ]:
batch[0]

In [ ]:
from pomegranate.hmm import  _check_input

In [ ]:
_check_input(batch[0][0], model)

In [ ]:
batch[0][0]

In [ ]:
entries_counter,  batches_counter, total_batches_counter

In [ ]:
isinstance(sequence_generator, BatchedDataGenerator)

In [ ]:
isinstance(sequence_generator, DataGenerator)

In [ ]:
data = sequence_generator.batches()

In [ ]:
initialization_batch_size = sequence_generator.batches_per_epoch
X_, labels_ = [], []
data = sequence_generator.batches()
counter = 0
for i in range(initialization_batch_size):
    counter += 1
    batch = next(data)
    X_.extend(batch[0])


In [ ]:
counter

In [ ]:

sequence_generator = DataGenerator(sample_X,batches_per_epoch=1, batch_size=50)

In [ ]:
sequence_generator

In [ ]:
len(sequence_generator)

In [ ]:
sequence_generator.batches_per_epoch

In [ ]:
sequence_generator.batch_size

In [ ]:
sequence_generator.finished

In [ ]:
sequence_generator.epoch

In [ ]:
sequence_generator.max_epoch

In [ ]:
sequence_generator.shape

In [ ]:
epoch_start = 0* sequence_generator.batches_per_epoch * sequence_generator.batch_size

In [ ]:
epoch_start, epoch_start + sequence_generator.batch_size

In [ ]:
sequence_generator.reset()

In [ ]:
batches_counter = 0
total_batches_counter = 0
epoch_counter = 0
entries_counter = 0
while True:
    epoch_counter += 1
    batches_counter = 0
    for batch in sequence_generator.batches():
        batch = batch
        entries_counter += len(batch[0])
        batches_counter+=1
        total_batches_counter += 1
    if sequence_generator.finished:
        break


In [ ]:
sequence_generator.batch_size

In [ ]:
entries_counter

In [ ]:
sequence_generator.batches_per_epoch

In [ ]:
batches_counter

In [ ]:
total_batches_counter

In [ ]:
epoch_counter

In [ ]:
sequence_generator.epoch

In [ ]:
sequence_generator.batches_per_epoch

In [ ]:
sequence_generator.batch_size

In [ ]:
sequence_generator.max_epoch

In [ ]:
sequence_generator.finished

In [ ]:
batch[0]

In [ ]:
X_, labels_ = [], []
labels = None
data = sequence_generator.batches()
for i in range(initialization_batch_size):
    batch = next(data)
    X_.extend(batch[0])
    if labels is not None:
        labels_.extend(batch[2])

In [ ]:
len(batch[0])

In [ ]:
i = 0

for batch in sequence_generator.batches():
    i = i+1

print(i)

In [ ]:
batch[0][0]

In [ ]:
sequence_generator.shape

In [ ]:
X_concat = np.concatenate(X_)
X_concat

In [ ]:
X_concat.ndim

In [ ]:
batch = next(data)

In [ ]:
[el for el in batch]